# 밑바닥부터 GPT 모델 구현하기 목차
* [Chapter 1 LLM 구조 구현하기](#chapter1)
* [Chapter 2 층 정규화로 활성화 정규화하기](#chapter2)

## Chapter 1 LLM 구조 구현하기 <a class="anchor" id="chapter1"></a>
1. GPT와 같은 LLM의 전체적인 구조는 아래 이미지와 같다.

    ![LLM 구조](image/04-01-structure2.png)

2. GTP-2 모델 구현 순서

     ![LLM 구조](image/04-01-dummy.png)




In [15]:
from importlib.metadata import version

print("맷플롯립 버전:", version("matplotlib"))
print("파이토치 버전:", version("torch"))
print("tiktoken 버전:", version("tiktoken"))

맷플롯립 버전: 3.10.6
파이토치 버전: 2.8.0
tiktoken 버전: 0.12.0


3. GPT 더미 모델 만들기

In [12]:
GPT_CONFIG_124M = {
    "vocab_size": 50257, # BPE 토크나이저에서 사용할 어휘사전 크기
    "context_length": 1024, # 문맥 길이 - 입력 토큰의 최대 개수
    "emb_dim": 768, # 임베딩 벡터의 차원 - 각 토큰을 768 차원 백터로 변환
    "n_heads": 12, # 멀티헤드 어텐션에서 사용할 헤드의 개수
    "n_layers": 12, # 트랜스포머 블록의 개수
    "dropout": 0.1, # 드롭아웃 비율 - 과대적합을 막기 위해 10%를 랜던하게 제외한다.
    "qkv_bias": False, # 쿼리, 키, 값 선형 변환에 바이어스 항을 사용할지 여부 - 훈련 시 False, 추론 시 True
}

In [1]:
import torch
import torch.nn as nn

# 나중에 실제 트랜스포머 블록으로 교체될 간단한 더미 클래스
class DummyTransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        
    def forward(self, x):
        return x
    

In [2]:
import torch
import torch.nn as nn

# 나중에 실제 층 정류화를 위한 층츠로 교체될 간단한 더미 클래스
class DummyLayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5):
        super().__init__()

    def forward(self, x):
        return x

In [10]:
import torch
import torch.nn as nn

class DummyGPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"]) # 토큰 임베딩 레이어
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"]) # 위치 임베딩 파라미터
        self.drop = nn.Dropout(cfg["dropout"])
        self.trf_blocks = nn.Sequential(*[DummyTransformerBlock(cfg) for _ in range(cfg["n_layers"])]) # 트랜스포머 블록들
        self.final_norm = DummyLayerNorm(cfg["emb_dim"]) # 층 정류화 레이어
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False) # 출력 헤드 레이어

    def forward(self, in_idx):
        batch_size, seq_length = in_idx.shape
        tok_emb = self.tok_emb(in_idx) # 토큰 임베딩
        pos_emb = self.pos_emb(torch.arange(0, seq_length, device=in_idx.device)) # 위치 임베딩
        x = tok_emb + pos_emb # 토큰 임베딩과 위치 임베딩의 합
        x = self.drop(x) # 토큰 임베딩과 위치 임베딩의 합에 드롭아웃 적용
        x = self.trf_blocks(x) # 트랜스포머 블록들 통과
        x = self.final_norm(x) # 층 정류화 적용
        
        # 출력 헤드 통과하여 로짓 계산
        #   - 로짓: 소프트맥스 함수나 시그모이드 함수를 사용하여 확률로 변화하기 전의 출력 값
        logits = self.out_head(x) 
        return logits
        


4. 토크나이저를 기반으로 GTP 모델로 데이터가 어떻게 입려되고 출력되는지 고수준으로 살펴보자
   - 이 과정을 구현하기 위해사 tiktokenizer 로 GPT 모델에서 사용할 2개의 텍스트로 구성된 배치를 토큰화 한다.

      ![더미 토큰](image/04-01-dummyToken.png)

In [6]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2") # GPT-2 토크나이저 로드

batch = []
txt1 = "Every effort moves you"
txt2 = "Every day holds a"

batch.append(torch.tensor(tokenizer.encode(txt1))) # 텍스트 1 토큰화 후 텐서로 변환하여 배치에 추가
print(batch)
batch.append(torch.tensor(tokenizer.encode(txt2))) # 텍스트 2 토큰화 후 텐서로 변환하여 배치에 추가
print(batch)
batch = torch.stack(batch, dim=0) # 배치 텐서로 스택킹
print(batch) # (배치 크기, 시퀀스 길이)
print(batch.shape) # (2, 6)

[tensor([6109, 3626, 6100,  345])]
[tensor([6109, 3626, 6100,  345]), tensor([6109, 1110, 6622,  257])]
tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])
torch.Size([2, 4])


5. 1억 2,400만 파라미터 크기의 DummyGPT 모델을 초기화하고 토큰화된 batch를 주입한다.

In [16]:
torch.manual_seed(123) # 재현성을 위해 시드 설정
model = DummyGPTModel(GPT_CONFIG_124M) # DummyGPT 모델 초기화
logits = model(batch) # 배치 입력을 모델에 주입하여 로짓 계산
print("출력 크기:", logits.shape) # (배치 크기, 시퀀스 길이, 어휘사전 크기)

# 출력 텐서는 2개의 텍스트 샘플에 해당하는 2개의 행을 갖는다.
#   - 각 텍스트 샘플은 4개의 토큰으로 구성된다.
#   - 각 토큰은 토크나이저의 어휘사전 크기에 해당하는 50,257차원의 벡터로 표현된다.
#   - 각 벡터의 값은 해당 토큰이 어휘사전의 각 단어에 속할 확률을 나타낸다.
#       - 예를 들어, logits[0, 0]는 첫 번째 텍스트 샘플의 첫 번째 토큰에 대한 로짓 벡터를 나타낸다.
#       - logits[0, 0, 100]는 첫 번째 텍스트 샘플의 첫 번째 토큰이 어휘사전의 100번째 단어에 속할 확률을 나타낸다.
#       - 이 값이 높을수록 해당 토큰이 100번째 단어에 속할 확률이 높다는 것을 의미한다.
#    - 후 처리코드를 구현할 때 이 50,257차원 벡터를 토큰 ID로 변환 후 다시 단어로 디코딩한다.
print("출력 예시:", logits)

출력 크기: torch.Size([2, 4, 50257])
출력 예시: tensor([[[-1.2034,  0.3201, -0.7130,  ..., -1.5548, -0.2390, -0.4667],
         [-0.1192,  0.4539, -0.4432,  ...,  0.2392,  1.3469,  1.2430],
         [ 0.5307,  1.6720, -0.4695,  ...,  1.1966,  0.0111,  0.5835],
         [ 0.0139,  1.6755, -0.3388,  ...,  1.1586, -0.0435, -1.0400]],

        [[-1.0908,  0.1798, -0.9484,  ..., -1.6047,  0.2439, -0.4530],
         [-0.7860,  0.5581, -0.0610,  ...,  0.4835, -0.0077,  1.6621],
         [ 0.3567,  1.2698, -0.6398,  ..., -0.0162, -0.1296,  0.3717],
         [-0.2407, -0.7349, -0.5102,  ...,  2.0057, -0.3694,  0.1814]]],
       grad_fn=<UnsafeViewBackward0>)


## Chapter 2 층 정규화로 활성화 정규화하기 <a class="anchor" id="chapter2"></a>
1. 많은 층을 가진 심층 신명망 훈련 시 그레이디언트 소실(vanishing gradient)과 폭주(exploding gradient) 문제가 발생한다.
    - 훈련 과정이 불안정해지고 신경망이 가중치를 효과적으로 조정하기 어렵게만든다.
    - 이 문제를 해결하기 위해 층 정규화(layer normalization) 기법이 제안되었다.
    - 층 정규화는 각 층의 입력을 정규화하여 그레이디언트가 안정적으로 전파되도록 돕는다.

2. 층 정규화의 핵심 아이디어는 신경망 층의 활성화(출력)를 평균 0이고 분산이 1이 되도록 조절하는 것이다.
    - 이렇게 하면 그레이디언트가 너무 작아지거나 너무 커지는 것을 방지할 수 있다.
    - 층 정규화는 미니배치 단위가 아닌 개별 샘플 단위로 정규화를 수행한다.
    - 이는 RNN과 같은 순환 신경망에서 특히 유용하다.

3. 층 정규화 예시
    - 5개의 입력과 6개의 출력을 가진 신경망 층에 2개의 입력 샘플적용

        ![층 정규화](image/04-02-nor2.png)

    - 이 신경망 층은 하나의 Linear 층과 하나의 ReLU 활성화 함수로 구성된다.
    - ReLU 활성화 함수는 음수를 0으로 바꾸고 양수는 그대로 통과시킨다.
    - 층 정규화를 적용하기 전에 평균과 분산을 확인

        ![dim](image/04-02-dim.png)    

In [18]:
torch.manual_seed(123) # 재현성을 위해 시드 설정
batch_example = torch.randn(2, 5) # 5개의 특성을 가진 2개의 샘플로 구성된 더미 배치 생성

# 선형 레이어와 ReLU 활성화 함수로 구성된 간단한 모델
#   - 5개의 입력 특성을 6개의 출력 특성으로 변환
layer = nn.Sequential(nn.Linear(5, 6), nn.ReLU()) 

out = layer(batch_example) # 더미 배치를 모델에 주입하여 출력 계산
print("출력 크기:", out.shape) # (배치 크기, 출력 특성 수)

# 첫 번째 행은 첫 번째 입력에 대한 층의 출력
# 두 번째 행은 두 번째 입력에 대한 층의 출력
print("출력 예시:", out) # 출력 예시

출력 크기: torch.Size([2, 6])
출력 예시: tensor([[0.2260, 0.3470, 0.0000, 0.2216, 0.0000, 0.0000],
        [0.2133, 0.2394, 0.0000, 0.5198, 0.3297, 0.0000]],
       grad_fn=<ReluBackward0>)


In [ ]:
# dim 매개변수
#   - 텐서의 계산이 수행되어야 하는 차원 지정
#   - 2차원 텐서에서 -1을 지정하는 것은 1을 사용하는 것과 같다.
# keepdim 매개변수
#   - dim 매개변수에 지정된 차원을 따라 텐서가 축소되는 연산이더라도 출력 텐서의 차원이 입력 텐서와 동일하게 유지
#   - 출력 텐서의 차원을 유지하여 후속 연산에서 차원 불일치 문제를 방지하는 데 유용하다.
#   - False : [0.1324, 0.2170]
#   - True  : [[0.1324], [0.2170]]
print("입력 크기: ", out.shape) # (2, 6)
mean = out.mean(dim=-1, keepdim=False)
print("keepdim=False 출력 크기:", mean.shape) # (2,)
print("keepdim=False 평균 예시:", mean)

mean = out.mean(dim=-1, keepdim=True) # 마지막 차원(특성 차원)을 따라 평균 계산
# 첫 번째 행은 첫 번째 입력에 대한 출력의 평균
# 두 번째 행은 두 번째 입력에 대한 출력의 평균
print("keepdim=False 출력 크기:", mean.shape) # (2,)
print("keepdim=True 평균 예시:", mean) # 평균 예시

var = out.var(dim=-1, keepdim=True) # 마지막 차원(특성 차원)을 따라 분산 계산
# 첫 번째 행은 첫 번째 입력에 대한 출력의 분산
# 두 번째 행은 두 번째 입력에 대한 출력의 분산
print("분산 예시:", var) # 분산 예시    

입력 크기:  torch.Size([2, 6])
keepdim=False 출력 크기: torch.Size([2])
keepdim=False 평균 예시: tensor([0.1324, 0.2170], grad_fn=<MeanBackward1>)
keepdim=False 출력 크기: torch.Size([2, 1])
keepdim=True 평균 예시: tensor([[0.1324],
        [0.2170]], grad_fn=<MeanBackward1>)
분산 예시: tensor([[0.0231],
        [0.0398]], grad_fn=<VarBackward0>)


In [29]:
torch.set_printoptions(sci_mode=False) # 소수점 4자리까지 출력하도록 설정
# 층 정규화
out_norm = (out - mean) / torch.sqrt(var) # 분산에 작은 상수를 더하여 수치적 안정성 확보
print("층 정규화 출력 예시:", out_norm) # 층 정규화 출력 예시

mean = out_norm.mean(dim=-1, keepdim=True) # 마지막 차원(특성 차원)을 따라 평균 계산
print("정규화 후 평균:", mean) # 정규화 후 평균 - 거의 0에 가까움

var = out_norm.var(dim=-1, keepdim=True) # 마지막 차원(특성 차원)을 따라 분산 계산
print("정규화 후 분산:", var) # 정규화 후 분산 - 거의 1에 가까움

층 정규화 출력 예시: tensor([[ 0.6143,  1.4108, -0.8731,  0.5857, -0.8731, -0.8731],
        [-0.0197,  0.1113, -1.0883,  1.5163,  0.5639, -1.0883]],
       grad_fn=<DivBackward0>)
정규화 후 평균: tensor([[-0.0014],
        [-0.0008]], grad_fn=<MeanBackward1>)
정규화 후 분산: tensor([[0.9996],
        [0.9997]], grad_fn=<VarBackward0>)


In [30]:
class LayerNorm(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.eps = 1e-5 # 0 나누기 방지를 위한 작은 상수 입실론
        self.scale = nn.Parameter(torch.ones(embed_dim)) # 학습 가능한 스케일 파라미터
        self.shift = nn.Parameter(torch.zeros(embed_dim)) # 학습 가능한 시프트 파라미터
        
    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True) # 마지막 차원(특성 차원)을 따라 평균 계산
        var = x.var(dim=-1, keepdim=True) # 마지막 차원(특성 차원)을 따라 분산 계산
        x_norm = (x - mean) / torch.sqrt(var + self.eps) # 층 정규화 계산
        return self.scale * x_norm + self.shift # 스케일과 시프트 적용하여 출력 반환

4. LayerNorm 모듈을 배치에 적용

In [ ]:
ln = LayerNorm(embed_dim=5) # 특성 차원이 5인 층 정규화 레이어 초기화
out_ln = ln(batch_example) # 더미 배치를 층 정규화 레이어에 주입하여 출력 계산
mean = out_ln.mean(dim=-1, keepdim=True) # 마지막 차원(특성 차원)을 따라 평균 계산
var = out_ln.var(dim=-1, keepdim=True) # 마지막 차원(특성 차원)을 따라 분산 계산
print("층 정규화 레이어 출력 예시:", out_ln) # 층 정규화 레이어 출력 예시
print("층 정규화 레이어 출력 평균:", mean) # 층 정규화 레이어 출력 평균 - 거의 0에 가까움
print("층 정규화 레이어 출력 분산:", var) # 층 정규화 레이어 출력 분산 - 거의 1에 가까움